# Aula: K-Means em Ciência de Dados — agrupando músicas do Spotify

**Disciplina:** ICD-004 Análise de Algoritmos · Fatec Jundiaí
**Professor:** Prof. Humberto Zanetti
**Data:** 24/11/2026

**Integrantes:**
- Wellington Torres
<!-- Novos integrantes: acrescente uma linha "- Nome completo" acima. -->

**Repositório:** [github.com/DataScience-Fatec-Projects/kmeans-musicas-spotify](https://github.com/DataScience-Fatec-Projects/kmeans-musicas-spotify)

> Como usar esta aula: leia de cima para baixo e execute as células em ordem
> (*Ambiente de execução → Executar tudo*). As células **🧪 Experimente** convidam você a mudar
> um parâmetro e rodar de novo.

## Sumário

- [⚙️ Configuração do ambiente](#⚙️-Configuração-do-ambiente)
- [I. Introdução e Contextualização](#I.-Introdução-e-Contextualização)
  - [I.1 Aprendizado não supervisionado](#I.1-Aprendizado-não-supervisionado)
  - [I.2 Clustering (agrupamento)](#I.2-Clustering-(agrupamento))
  - [I.3 Como o K-Means funciona — implementado do zero](#I.3-Como-o-K-Means-funciona-—-implementado-do-zero)
  - [I.4 Quanto custa? Complexidade do K-Means](#I.4-Quanto-custa?-Complexidade-do-K-Means)
  - [I.5 Como avaliar: inércia, Método do Cotovelo e silhueta](#I.5-Como-avaliar:-inércia,-Método-do-Cotovelo-e-silhueta)
- [II. Ferramentas de Trabalho (Scikit-learn e Python)](#II.-Ferramentas-de-Trabalho-(Scikit-learn-e-Python))
  - [II.1 Bibliotecas e importações](#II.1-Bibliotecas-e-importações)
  - [II.2 O KMeans do scikit-learn](#II.2-O-KMeans-do-scikit-learn)
- [III. Escolha do Tema e Preparação dos Dados](#III.-Escolha-do-Tema-e-Preparação-dos-Dados)
  - [III.1 Tema: por que agrupar músicas?](#III.1-Tema:-por-que-agrupar-músicas?)
  - [III.2 O dataset: origem e dicionário de variáveis](#III.2-O-dataset:-origem-e-dicionário-de-variáveis)
  - [III.3 Carregamento e limpeza](#III.3-Carregamento-e-limpeza)
  - [III.4 Escolha das features](#III.4-Escolha-das-features)
  - [III.5 Padronização com StandardScaler](#III.5-Padronização-com-StandardScaler)
- [IV. Implementação e Aplicação Prática](#IV.-Implementação-e-Aplicação-Prática)
  - [IV.1 Definindo k: Método do Cotovelo (e silhueta)](#IV.1-Definindo-k:-Método-do-Cotovelo-(e-silhueta))
  - [IV.2 Treinando o K-Means](#IV.2-Treinando-o-K-Means)
  - [IV.3 Visualizando a separação dos clusters](#IV.3-Visualizando-a-separação-dos-clusters)
  - [IV.4 Centroides em relação aos pontos](#IV.4-Centroides-em-relação-aos-pontos)
- [V. Análise e conclusão](#V.-Análise-e-conclusão)
  - [V.1 Perfil de cada cluster](#V.1-Perfil-de-cada-cluster)
  - [V.2 Dando nome aos grupos](#V.2-Dando-nome-aos-grupos)
  - [V.3 Clusters × gêneros (com destaque para a música brasileira)](#V.3-Clusters-×-gêneros-(com-destaque-para-a-música-brasileira))
  - [V.4 Limitações do K-Means](#V.4-Limitações-do-K-Means)
  - [V.5 Próximos passos](#V.5-Próximos-passos)
- [Referências](#Referências)

> No Colab, o painel **Índice** (ícone ☰ à esquerda) também mostra esta estrutura.

## ⚙️ Configuração do ambiente

Esta célula prepara tudo o que o resto da aula usa. Ela fica aqui no começo, e não na Seção II,
porque a Seção I já desenha gráficos e roda um K-Means de exemplo. A Seção II explica cada
biblioteca com calma.

O que ela faz:

1. **Importa as bibliotecas**: `numpy` e `pandas` (dados), `matplotlib` e `seaborn` (gráficos) e
   as partes do `scikit-learn` usadas na aula (padronização, K-Means, métricas, PCA e dados de
   exemplo).
2. **Fixa a semente aleatória** `RANDOM_STATE = 42`. O K-Means e as amostragens sorteiam números;
   com a mesma semente, quem rodar a aula de novo vê **exatamente** os mesmos resultados.
3. **Define as constantes da análise**: o endereço do dataset, as 9 *features* de áudio usadas
   no agrupamento, os gêneros que vamos destacar e o tamanho das amostras usadas nas partes mais
   caras (silhueta e gráficos de dispersão).
4. **Escolhe o estilo dos gráficos** e **imprime as versões** do Python e das bibliotecas, para
   que dê para comparar se algum resultado sair diferente.

In [ ]:
import sys

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import sklearn
from sklearn.cluster import KMeans
from sklearn.datasets import make_blobs
from sklearn.decomposition import PCA
from sklearn.metrics import adjusted_rand_score, silhouette_score
from sklearn.preprocessing import StandardScaler

# Semente única de toda a aula (constituição IV): não crie outra.
RANDOM_STATE = 42

# Dataset público no Hugging Face (lido direto da internet, sem upload).
URL_DATASET = (
    "https://huggingface.co/datasets/maharshipandya/spotify-tracks-dataset"
    "/resolve/main/dataset.csv"
)

# As 9 features contínuas de áudio que entram no K-Means (FR-035).
FEATURES = [
    "danceability", "energy", "loudness", "speechiness", "acousticness",
    "instrumentalness", "liveness", "valence", "tempo",
]

# Gêneros em destaque na Seção V (FR-034) e gêneros internacionais de contraste.
GENEROS_BR = ["brazil", "mpb", "sertanejo", "pagode", "samba", "forro", "funk"]
GENEROS_CONTRASTE = ["classical", "metal", "sleep", "hip-hop", "edm", "comedy"]

# Amostras para as etapas caras (research R3).
TAMANHO_AMOSTRA_SILHUETA = 10_000  # a silhueta exata é O(n²)
TAMANHO_AMOSTRA_GRAFICOS = 6_000   # dispersão legível, sem virar um borrão

# Estilo padrão dos gráficos.
sns.set_theme(style="whitegrid", palette="tab10")
plt.rcParams.update({"figure.figsize": (9, 5), "figure.dpi": 100, "axes.titleweight": "bold"})

print(f"Python       {sys.version.split()[0]}")
for nome, modulo in [("pandas", pd), ("numpy", np), ("scikit-learn", sklearn),
                     ("matplotlib", matplotlib), ("seaborn", sns)]:
    print(f"{nome:<12} {modulo.__version__}")

Acima estão as versões usadas nesta execução. No Colab elas mudam de tempos em tempos; se algum
número da aula sair um pouco diferente do texto, compare primeiro estas versões.

## I. Introdução e Contextualização

### I.1 Aprendizado não supervisionado

> 🚧 **Em construção** — ver T021 ([#9](https://github.com/DataScience-Fatec-Projects/kmeans-musicas-spotify/issues/9)).

### I.2 Clustering (agrupamento)

> 🚧 **Em construção** — ver T021 ([#9](https://github.com/DataScience-Fatec-Projects/kmeans-musicas-spotify/issues/9)).

### I.3 Como o K-Means funciona — implementado do zero

> 🚧 **Em construção** — ver T022 ([#10](https://github.com/DataScience-Fatec-Projects/kmeans-musicas-spotify/issues/10)), T023 ([#11](https://github.com/DataScience-Fatec-Projects/kmeans-musicas-spotify/issues/11)).

#### 🧪 Experimente

> 🚧 **Em construção** — ver T024 ([#12](https://github.com/DataScience-Fatec-Projects/kmeans-musicas-spotify/issues/12)).

Parâmetro a mudar: `K_BRINQUEDO` e `SEMENTE_INICIAL`. O que observar: inicializações diferentes mudam as iterações e às vezes o resultado (mínimo local).

### I.4 Quanto custa? Complexidade do K-Means

> 🚧 **Em construção** — ver T025 ([#13](https://github.com/DataScience-Fatec-Projects/kmeans-musicas-spotify/issues/13)).

### I.5 Como avaliar: inércia, Método do Cotovelo e silhueta

> 🚧 **Em construção** — ver T026 ([#14](https://github.com/DataScience-Fatec-Projects/kmeans-musicas-spotify/issues/14)).

## II. Ferramentas de Trabalho (Scikit-learn e Python)

### II.1 Bibliotecas e importações

> 🚧 **Em construção** — ver T033 ([#16](https://github.com/DataScience-Fatec-Projects/kmeans-musicas-spotify/issues/16)).

### II.2 O KMeans do scikit-learn

> 🚧 **Em construção** — ver T034 ([#17](https://github.com/DataScience-Fatec-Projects/kmeans-musicas-spotify/issues/17)).

## III. Escolha do Tema e Preparação dos Dados

### III.1 Tema: por que agrupar músicas?

O Spotify tem dezenas de milhões de faixas. Ninguém consegue ouvir todas e anotar à mão se cada
uma é "calma", "dançante" ou "pesada". Mesmo assim, é exatamente esse tipo de informação que o
app usa o tempo todo:

- **Playlists automáticas**: "Para focar", "Treino", "Relaxar". Cada uma junta músicas que *soam*
  parecidas, mesmo sendo de artistas e gêneros diferentes.
- **Recomendação**: se você gosta de uma faixa, as faixas que estão no mesmo grupo dela são boas
  candidatas para tocar em seguida.
- **Descoberta de "climas"**: um samba e um pop podem ter a mesma energia dançante, e uma faixa
  de música clássica pode estar mais perto de um *lo-fi* do que de outra clássica. Os gêneros
  não contam essa história inteira.

Para cada faixa, o Spotify calculou números que descrevem o som: o quanto ela é dançante,
enérgica, acústica, alegre, falada etc. Não existe uma resposta certa dizendo a que grupo cada
música pertence. É um problema de **aprendizado não supervisionado** (Seção I): damos ao K-Means
só as características e deixamos que ele descubra os grupos.

> **A pergunta desta aula:** que perfis de som existem nas músicas do Spotify, e onde a música
> brasileira se encaixa neles?

O caminho até o modelo, nesta seção:

1. **III.2** conhecer o dataset e o que significa cada coluna;
2. **III.3** carregar e limpar os dados;
3. **III.4** escolher quais colunas entram no agrupamento;
4. **III.5** colocar todas as colunas na mesma escala.

### III.2 O dataset: origem e dicionário de variáveis

**Origem.** Usamos o [*Spotify Tracks Dataset*](https://huggingface.co/datasets/maharshipandya/spotify-tracks-dataset), publicado no Hugging Face por
**Maharshi Pandya** sob a licença **BSD**. Os dados foram coletados com a API do Spotify e
trazem **114.000 linhas**: 1.000 faixas para cada um dos **114 gêneros**.

> ⚠️ Em novembro de 2024, o Spotify deixou de oferecer as *audio features* (dançabilidade,
> energia etc.) na sua API para novos aplicativos. Hoje não dá para coletar esses números de
> novo, e por isso trabalhamos com este retrato estático, lido direto do Hugging Face.

**Dicionário de variáveis.** Cada linha é uma combinação (faixa, gênero). As faixas de valores
abaixo são as dos dados brutos, antes da limpeza da III.3.

| Coluna | Significado | Valores | Entra no agrupamento? |
|---|---|---|---|
| `track_id` | identificador da faixa no Spotify | texto | não: é a chave |
| `artists` | artista(s), separados por `;` | texto | não: usado nos exemplos da V |
| `album_name` | nome do álbum | texto | não |
| `track_name` | nome da faixa | texto | não: usado nos exemplos da V |
| `popularity` | popularidade calculada pelo Spotify (quantidade e recência de execuções) | 0 a 100 | não: contexto, não é som |
| `duration_ms` | duração em milissegundos | 0 a ≈ 5,2 milhões (≈ 87 min) | não: contexto, não é som |
| `explicit` | a letra tem conteúdo explícito? | `True`/`False` | não: contexto, não é som |
| `danceability` | o quanto a faixa é boa para dançar (ritmo estável, batida forte) | 0 a 0,985 | **sim** |
| `energy` | intensidade percebida: rápida, alta, barulhenta | 0 a 1 | **sim** |
| `key` | tom da música (0 = Dó, 1 = Dó♯, …, 11 = Si) | 0 a 11 | não: categórica |
| `loudness` | volume médio, em decibéis (dB) | −49,5 a +4,5 | **sim** |
| `mode` | modo: 1 = maior, 0 = menor | 0 ou 1 | não: categórica |
| `speechiness` | presença de fala (perto de 1 = podcast, *stand-up*) | 0 a 0,965 | **sim** |
| `acousticness` | confiança de que a faixa é acústica | 0 a 0,996 | **sim** |
| `instrumentalness` | probabilidade de a faixa não ter voz | 0 a 1 | **sim** |
| `liveness` | probabilidade de ter sido gravada ao vivo, com plateia | 0 a 1 | **sim** |
| `valence` | positividade: alto = alegre, eufórica; baixo = triste, tensa | 0 a 0,995 | **sim** |
| `tempo` | andamento, em batidas por minuto (BPM) | 0 a 243 | **sim** (os zeros são inválidos, III.3) |
| `time_signature` | compasso estimado (3 = 3/4, 4 = 4/4…) | 0 a 5 | não: categórica |
| `track_genre` | gênero atribuído pela fonte | 114 gêneros | não: reservada para validar os grupos na V.3 |

A III.4 explica, uma por uma, por que essas 9 colunas entram e as outras ficam de fora.

### III.3 Carregamento e limpeza

**Carregando os dados.** O arquivo CSV (≈ 20 MB) é lido direto do endereço guardado em
`URL_DATASET`, sem upload e sem login, então a aula roda igual em qualquer Colab.

Dois detalhes do código:

- `index_col=0`: a primeira coluna do arquivo é só um número de linha, sem nome. Ela vira o
  índice da tabela e não uma coluna de dados.
- `try`/`except`: se a internet ou o Hugging Face falharem, aparece uma mensagem em português
  com o link da página do dataset, em vez de só um erro técnico.

In [ ]:
try:
    df_bruto = pd.read_csv(URL_DATASET, index_col=0)
except Exception as erro:
    print("Não foi possível baixar o dataset.")
    print(f"Motivo: {erro}")
    print("Confira a conexão com a internet ou baixe pela página original:")
    print("https://huggingface.co/datasets/maharshipandya/spotify-tracks-dataset")
    raise

linhas, colunas = df_bruto.shape
print(f"df_bruto: {linhas:,} linhas × {colunas} colunas".replace(",", "."))
df_bruto.head()

Vieram **114.000 linhas e 20 colunas**, como a página do dataset descreve. As primeiras linhas
mostram as três partes da tabela: a identificação da faixa (`track_id`, artista, álbum, nome),
os números que descrevem o som (`danceability` até `tempo`) e, na última coluna, o gênero.

Agora vamos ver o **tipo** de cada coluna e se alguma tem valores faltando.

In [ ]:
df_bruto.info()

Os tipos batem com o dicionário da III.2: as 9 *features* de áudio são números decimais
(`float64`), `key`, `mode` e `time_signature` são inteiros e `explicit` é verdadeiro/falso.

Repare na coluna **Non-Null Count**: `artists`, `album_name` e `track_name` têm **113.999**
valores, um a menos que as outras. Existe uma linha com dados faltando, e é por ela que a limpeza
começa.

**Limpando os dados.** Três regras, aplicadas nesta ordem:

1. **Linhas com valor ausente** são removidas. É a linha sem artista, álbum e nome que a
   `info()` mostrou. Sem nome, não daria para usá-la nos exemplos da Seção V.
2. **Linhas idênticas** (todas as 20 colunas iguais) são cópias acidentais e são removidas.
3. **`tempo` = 0** é removido. Nenhuma música tem 0 batidas por minuto: esse valor aparece em
   faixas de silêncio, ruído ou só fala, em que o Spotify não conseguiu medir o andamento. É um
   valor **inválido**, não um valor pequeno, e atrapalharia as distâncias.

O resultado se chama `df_pares`, porque cada linha ainda é um par (faixa, gênero).

In [ ]:
sem_ausentes = df_bruto.dropna()
sem_identicas = sem_ausentes.drop_duplicates()
df_pares = sem_identicas[sem_identicas["tempo"] > 0].reset_index(drop=True)

limpeza = pd.DataFrame({
    "regra": ["1. valor ausente", "2. linha idêntica", "3. tempo = 0"],
    "linhas removidas": [
        len(df_bruto) - len(sem_ausentes),
        len(sem_ausentes) - len(sem_identicas),
        len(sem_identicas) - len(df_pares),
    ],
})
print(f"df_pares: {len(df_pares):,} pares (faixa, gênero)".replace(",", "."))
limpeza

A limpeza removeu **1 linha** com ausentes, **450 linhas idênticas** e **157 faixas com
`tempo` = 0**. Sobraram **113.392 pares**. Perdemos menos de 0,6% dos dados.

**Uma faixa, vários gêneros.** O dataset tem 1.000 linhas *por gênero*, e a mesma faixa pode
aparecer em mais de um: uma música pode estar em `mpb` e também em `brazil`. Quantos gêneros
cada faixa tem?

In [ ]:
generos_por_faixa = df_pares.groupby("track_id")["track_genre"].nunique()
(generos_por_faixa.value_counts().sort_index()
    .rename_axis("gêneros da faixa").to_frame("faixas"))

A maioria das faixas (73.284) aparece em um gênero só, mas **16.299 faixas aparecem em dois ou
mais**, e uma delas chega a 9 gêneros. Juntas, essas repetições somam 23.809 linhas a mais.

**Por que isso importa para o K-Means?** O centroide de um grupo é a **média** dos pontos do
grupo. Se treinarmos com `df_pares`, uma faixa que está em 9 gêneros entra 9 vezes nessa média e
puxa o centroide para perto dela, só porque a fonte a classificou em mais gêneros. Nenhuma faixa
deveria pesar mais que outra.

Por isso a base de treino terá **uma linha por faixa**. Antes, conferimos com um `assert` (uma
checagem que interrompe o notebook se falhar) que as cópias de uma faixa têm exatamente os
mesmos valores de áudio, para garantir que não perdemos informação ao ficar com uma delas.

In [ ]:
faixas_divergentes = (df_pares.groupby("track_id")[FEATURES].nunique() > 1).any(axis=1).sum()
assert faixas_divergentes == 0, f"{faixas_divergentes} faixas têm áudio diferente entre as cópias"

df_faixas = (df_pares.drop_duplicates("track_id")
             .drop(columns="track_genre")
             .reset_index(drop=True))
print(f"Faixas com áudio diferente entre as cópias: {faixas_divergentes}")
print(f"df_faixas: {len(df_faixas):,} faixas únicas (base de treino)".replace(",", "."))
print(f"df_pares:  {len(df_pares):,} pares, com {df_pares['track_genre'].nunique()} gêneros "
      "(guardado para a análise por gênero da V.3)".replace(",", "."))

O `assert` passou: **nenhuma** faixa tem valores de áudio diferentes entre as cópias. A base de
treino `df_faixas` tem **89.583 faixas únicas**.

Duas decisões ficam registradas aqui:

- `df_faixas` **não tem a coluna `track_genre`**. Ela foi retirada de propósito: como cada faixa
  pode ter vários gêneros, guardar só um seria arbitrário, e o gênero não deve chegar perto do
  treino (III.4).
- `df_pares` continua guardado, com os 114 gêneros. Na Seção V, cada faixa recebe o seu cluster,
  e esse rótulo volta para **todas** as linhas dela em `df_pares`. Assim nenhum gênero perde
  faixas na análise.

### III.4 Escolha das features

> 🚧 **Em construção** — ver T011 ([#23](https://github.com/DataScience-Fatec-Projects/kmeans-musicas-spotify/issues/23)).

### III.5 Padronização com StandardScaler

> 🚧 **Em construção** — ver T012 ([#24](https://github.com/DataScience-Fatec-Projects/kmeans-musicas-spotify/issues/24)).

## IV. Implementação e Aplicação Prática

### IV.1 Definindo k: Método do Cotovelo (e silhueta)

> 🚧 **Em construção** — ver T013 ([#26](https://github.com/DataScience-Fatec-Projects/kmeans-musicas-spotify/issues/26)), T014 ([#27](https://github.com/DataScience-Fatec-Projects/kmeans-musicas-spotify/issues/27)), T015 ([#28](https://github.com/DataScience-Fatec-Projects/kmeans-musicas-spotify/issues/28)).

### IV.2 Treinando o K-Means

> 🚧 **Em construção** — ver T016 ([#29](https://github.com/DataScience-Fatec-Projects/kmeans-musicas-spotify/issues/29)).

#### 🧪 Experimente

> 🚧 **Em construção** — ver T017 ([#30](https://github.com/DataScience-Fatec-Projects/kmeans-musicas-spotify/issues/30)).

Parâmetro a mudar: `K_TESTE`. O que observar: como os perfis se dividem ou se fundem com mais ou menos clusters.

### IV.3 Visualizando a separação dos clusters

> 🚧 **Em construção** — ver T018 ([#31](https://github.com/DataScience-Fatec-Projects/kmeans-musicas-spotify/issues/31)).

### IV.4 Centroides em relação aos pontos

> 🚧 **Em construção** — ver T019 ([#32](https://github.com/DataScience-Fatec-Projects/kmeans-musicas-spotify/issues/32)).

#### 🧪 Experimente

> 🚧 **Em construção** — ver T020 ([#33](https://github.com/DataScience-Fatec-Projects/kmeans-musicas-spotify/issues/33)).

Parâmetro a mudar: `USAR_PADRONIZACAO = True/False`. O que observar: sem padronizar, `tempo` e `loudness` dominam a distância e os clusters mudam.

## V. Análise e conclusão

### V.1 Perfil de cada cluster

> 🚧 **Em construção** — ver T027 ([#35](https://github.com/DataScience-Fatec-Projects/kmeans-musicas-spotify/issues/35)).

### V.2 Dando nome aos grupos

> 🚧 **Em construção** — ver T028 ([#36](https://github.com/DataScience-Fatec-Projects/kmeans-musicas-spotify/issues/36)).

### V.3 Clusters × gêneros (com destaque para a música brasileira)

> 🚧 **Em construção** — ver T029 ([#37](https://github.com/DataScience-Fatec-Projects/kmeans-musicas-spotify/issues/37)), T030 ([#38](https://github.com/DataScience-Fatec-Projects/kmeans-musicas-spotify/issues/38)).

#### 🧪 Experimente

> 🚧 **Em construção** — ver T031 ([#39](https://github.com/DataScience-Fatec-Projects/kmeans-musicas-spotify/issues/39)).

Parâmetro a mudar: `GENEROS_EXTRA`. O que observar: onde caem os gêneros que você escolher.

### V.4 Limitações do K-Means

> 🚧 **Em construção** — ver T032 ([#40](https://github.com/DataScience-Fatec-Projects/kmeans-musicas-spotify/issues/40)).

### V.5 Próximos passos

> 🚧 **Em construção** — ver T032 ([#40](https://github.com/DataScience-Fatec-Projects/kmeans-musicas-spotify/issues/40)).

## Referências

> 🚧 **Em construção** — ver T035 ([#42](https://github.com/DataScience-Fatec-Projects/kmeans-musicas-spotify/issues/42)).